# 1. Build and Train a GPT

Notebook 00 showed *where* low precision pays, using a stack of `Linear` layers. Now we
build the real thing: a GPT language model, trained from scratch on short stories, that you
can actually talk to at the end.

Everything is written out in this notebook. There is no model library to import and no
hidden code -- by the end of section 1.4 you will have typed a complete GPT in about forty
lines, and every later notebook changes one piece of it.

## Objectives

- Understand what a language model does, in plain terms
- Write a transformer block: attention, an MLP, and two residual connections
- Assemble it into a working GPT with embeddings and an output head
- Train it on TinyStories and read what it writes
- Establish the baseline speed that notebooks 02 and 03 are measured against

## Requirements

- Notebook 00 first -- it explains FP8, and why size decides whether it helps
- The dataset
    * Tokenizer - GPT-2's BPE vocabulary and merge rules
        * License: MIT
        * Files:
            * [tokenizer.json](https://huggingface.co/gpt2/resolve/main/tokenizer.json)
    * Training data - Dataset containing synthetically generated short stories that only use a small vocabulary
        * Licnese: CDLA-Sharing-1.0
        * Files:
            * [TinyStoriesV2-GPT4-train.txt](https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt)
            * [TinyStoriesV2-GPT4-valid.txt](https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt)
- An NVIDIA GPU with ~10 GB free

## Working through this on your own

Run the cells in order. Training in 1.5 takes about a minute; everything else is seconds.

In [2]:
%%bash
mkdir -p data

# Tokenizer
wget -c -nv -P data https://huggingface.co/gpt2/resolve/main/tokenizer.json

# Training data
wget -c -nv -P data https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
# Validation data
wget -c -nv -P data https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt

# Split, tokenize, and write into blocks
python prepare_data.py

train:
  tokenizing -> train.bin
      300.0M tokens  done in 72s        
val:
  tokenizing -> val.bin
        5.5M tokens  done in 1s        

ready: 300M train / 5M val tokens in data/


In [1]:
import logging
import math
import time
import warnings
from dataclasses import dataclass

# Quiet the container's import-time deprecation notices (before importing TE).
logging.getLogger("torch._library.opaque_object").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from helpers import summary                        # hardware report only
from helpers import get_tokenizer     # GPT-2's BPE vocabulary

torch.manual_seed(1337)
print(summary())

GPU              : NVIDIA H100 NVL
compute capability: sm_90
SMs / memory     : 132 SMs, 99.9 GB
measured BF16 peak: 534 TFLOP/s (dense matmul)
measured FP8 peak : 894 TFLOP/s (dense matmul)

Transformer Engine FP8 recipe support:
  delayed  yes
  current  yes
  block    yes
  block_rowwise yes
  mxfp8    no   (Device compute capability 10.0 or higher required for MXFP8 execution.)
  nvfp4    no   (Device compute capability 10.0 or higher required for NVFP4 execution.)


## 1.1 What does a language model actually do?

One thing: **given some text, predict the next token.**

That is the whole training objective. Everything else -- writing stories, answering
questions, summarizing -- is that one ability applied repeatedly, feeding each prediction
back in as input.

A *token* is a chunk of text, usually a common word or a piece of one. The model never sees
letters; it sees integers, one per token, and its output is a score for every token in the
vocabulary. Training means adjusting the weights until the correct next token tends to get
the highest score.

## 1.2 The data

We use [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories): short,
simple stories written with a small vocabulary. The point is that a small model trained for
a minute produces *recognizable stories*. On general web text the same model would produce
fluent-sounding nonsense, and you could not tell a real problem from ordinary undertraining.

The first cell in this notebook downloads the stories from Hugging Face, along with GPT-2's vocabulary, and then tokenizes everything into two flat files of integers.

In [2]:
enc = get_tokenizer("data")

sample = "Once upon a time, Mia built a small robot."
ids = enc.encode(sample)
print(f"text   : {sample}")
print(f"tokens : {ids}")
print(f"back   : {enc.decode(ids)}")
print(f"\n{len(sample)} characters -> {len(ids)} tokens\n")

# Common words are single tokens; rarer ones get split up.
for tok in ids:
    print(f"  {tok:6d} -> {enc.decode([tok])!r}")

text   : Once upon a time, Mia built a small robot.
tokens : [7454, 2402, 257, 640, 11, 32189, 3170, 257, 1402, 9379, 13]
back   : Once upon a time, Mia built a small robot.

42 characters -> 11 tokens

    7454 -> 'Once'
    2402 -> ' upon'
     257 -> ' a'
     640 -> ' time'
      11 -> ','
   32189 -> ' Mia'
    3170 -> ' built'
     257 -> ' a'
    1402 -> ' small'
    9379 -> ' robot'
      13 -> '.'


The token files are one long stream of these integers -- 300 million of them for
training. To make a batch we pick random starting points and cut fixed-length windows.

The targets are the same window shifted one position left, because the goal at every
position is "predict what comes next".

In [3]:
@dataclass
class Config:
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768           # hidden size: the width of the residual stream
    seq_len: int = 1024      # context length: how many tokens the model sees at once
    vocab_size: int = 50304     # GPT-2 has 50257; see the note below

data = {}
data['train'] = np.memmap(f"data/train.bin", dtype=np.uint16, mode="r")
data['val'] = np.memmap(f"data/val.bin", dtype=np.uint16, mode="r")
def get_batch(split, cfg, batch_size, device="cuda"):
    """A random batch of (inputs, targets) from the token stream."""
    D = data[split]
    i = torch.randint(len(D) - cfg.seq_len - 1, (batch_size,))
    x = torch.stack([torch.from_numpy(D[j:j + cfg.seq_len].astype(np.int64)) for j in i])
    y = torch.stack([torch.from_numpy(D[j + 1:j + 1 + cfg.seq_len].astype(np.int64)) for j in i])
    return x.to(device), y.to(device)

cfg = Config()
x, y = get_batch("train", cfg, batch_size=4)
print(f"inputs  {tuple(x.shape)}   targets {tuple(y.shape)}\n")
print("first sequence starts:", enc.decode(x[0, :12].tolist()))
print("its targets start    :", enc.decode(y[0, :12].tolist()))

inputs  (4, 1024)   targets (4, 1024)

first sequence starts:  lots of fun. But then, when Mary suggested they play
its targets start    :  of fun. But then, when Mary suggested they play on


> **Why `vocab_size = 50304` when GPT-2 has 50257 tokens?**
> Because 50304 is a multiple of 128 and 50257 is not. The output layer is a matrix
> multiply with the vocabulary as one of its dimensions, and notebook 00 section 0.6 showed
> what happens to shapes that are not multiples of 16: at best they run slower, at worst
> FP8 refuses them. The 47 extra rows are unused tokens that cost a little memory and buy a
> tensor-core-friendly shape. This is the cheapest alignment win in the workshop.

## 1.3 The transformer block

A GPT is a stack of identical blocks. Each block does two things, and wraps each one in a
**residual connection** -- `x + something(x)` rather than `something(x)`:

```
x = x + attention(norm(x))     "look at other tokens and gather information"
x = x + mlp(norm(x))           "think about what you gathered"
```

The residual connections are what make deep stacks trainable: each block *adjusts* the
running representation rather than replacing it, so gradients have a short path back to
every layer.

**Attention** lets each position look at earlier positions. Every token produces three
vectors -- a **query** (what am I looking for?), a **key** (what do I offer?) and a
**value** (what will I pass on?). Each query is compared against every key to decide how
much attention to pay, and the result is a weighted average of the values.

It is *causal*: position `i` may only look at positions `<= i`. A model allowed to see the
future could cheat by reading the answer.

In [4]:
class TorchBlock(nn.Module):
    """One transformer block, in plain PyTorch."""

    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.qkv  = nn.Linear(cfg.n_embd, 3 * cfg.n_embd)   # queries, keys, values at once
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd)       # mix the heads back together
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.fc_1 = nn.Linear(cfg.n_embd, 4 * cfg.n_embd)   # the MLP widens 4x...
        self.fc_2 = nn.Linear(4 * cfg.n_embd, cfg.n_embd)   # ...then comes back

    def attention(self, qkv):
        B, T, C3 = qkv.shape
        C = C3 // 3
        # Split into q, k, v and give each head its own slice: (B, heads, T, head_dim).
        q, k, v = (z.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
                   for z in qkv.split(C, dim=2))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

    def forward(self, x):
        x = x + self.attention(self.qkv(self.ln_1(x)))
        return x + self.fc_2(F.gelu(self.fc_1(self.ln_2(x)), approximate="tanh"))


block = TorchBlock(cfg)
print(block)
print(f"\nparameters in one block: {sum(p.numel() for p in block.parameters())/1e6:.1f}M")

TorchBlock(
  (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  (qkv): Linear(in_features=768, out_features=2304, bias=True)
  (proj): Linear(in_features=768, out_features=768, bias=True)
  (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  (fc_1): Linear(in_features=768, out_features=3072, bias=True)
  (fc_2): Linear(in_features=3072, out_features=768, bias=True)
)

parameters in one block: 7.1M


`F.scaled_dot_product_attention` does the four steps of attention -- score, mask,
softmax, weighted average -- in one fused kernel. Written out by hand it would build the
full `T x T` score matrix in memory, which is what makes long contexts expensive.

Notice there are **four** matrix multiplies per block: `qkv`, `proj`, `fc_1`, `fc_2`. Those
are where nearly all the arithmetic is, and they are exactly the layers notebooks 02 and 03
replace. Attention's own matmuls are a smaller share.

## 1.4 The whole model

Wrapping the stack of blocks:

- **Token embeddings** turn each token id into a vector
- **Position embeddings** add "where am I in the sequence", since attention alone has no
  notion of order
- **The head** turns the final vector back into one score per vocabulary token
- **Weight tying** makes the head reuse the token-embedding matrix -- it saves ~39M
  parameters and slightly improves quality

In [5]:
class MiniGPT(nn.Module):
    """A GPT, complete. `Block` is the one piece later notebooks swap."""

    def __init__(self, cfg, Block=TorchBlock):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)   # token embeddings
        self.wpe = nn.Embedding(cfg.seq_len, cfg.n_embd)   # position embeddings
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.wte.weight                    # weight tying
        self.init_weights()

    # Layers that write back into the residual stream get a SMALLER initialization, scaled
    # by 1/sqrt(2 * n_layer). Without it the residual stream's variance grows with depth and
    # deep models become hard to train. Each block type spells these layers differently, so
    # match every suffix -- a name we fail to match is silently left at the wrong scale.
    RESIDUAL = ("proj.weight", "fc_2.weight", "fc2_weight")

    def init_weights(self):
        small = 0.02 / math.sqrt(2 * self.cfg.n_layer)
        for name, p in self.named_parameters():
            if p.dim() < 2:
                # Biases. nn.Linear seeds them randomly while te.Linear zeroes them, so we
                # set them explicitly -- otherwise the block types start from different
                # models and the comparison is measuring two things at once. (GPT-2 uses
                # zero biases anyway.) LayerNorm gains are already ones; leave them.
                if name.endswith("bias") and "layer_norm" not in name and "ln" not in name:
                    nn.init.zeros_(p)
                continue
            nn.init.normal_(p, std=small if name.endswith(self.RESIDUAL) else 0.02)

    def forward(self, idx, targets=None):
        pos = torch.arange(idx.size(1), device=idx.device)
        x = self.wte(idx) + self.wpe(pos)
        for b in self.blocks:
            x = b(x)
        logits = self.head(self.ln_f(x))
        if targets is None:
            return logits, None
        # The objective, in one line: at every position, predict the next token.
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss


model = MiniGPT(cfg).cuda()
n_resid = sum(1 for n, p in model.named_parameters()
              if p.dim() >= 2 and n.endswith(MiniGPT.RESIDUAL))
print(f"parameters       : {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print(f"residual-scaled  : {n_resid} tensors (expected {2 * cfg.n_layer}: two per block)")

_, loss = model(*get_batch("train", cfg, 4))
print(f"\nloss before training: {loss.item():.3f}")
print(f"random guessing would be: {math.log(cfg.vocab_size):.3f}")

parameters       : 124.5M
residual-scaled  : 24 tensors (expected 24: two per block)

loss before training: 10.941
random guessing would be: 10.826


The untrained loss matches `log(vocab_size)` because the model starts out assigning
roughly equal probability to every token. That is a useful sanity check: if your loss starts
far from it, something is wrong with the initialization or the data.

> **That `RESIDUAL` tuple is load-bearing.** The scaled initialization is applied by
> matching parameter *names*, and notebooks 02 and 03 introduce blocks that name the same
> layers differently (`fc_2.weight` becomes `ln_mlp.fc2_weight`). A missed name leaves that
> tensor at the wrong scale, and the model trains to a slightly worse loss -- which looks
> exactly like "the new block type hurts quality" and is not. The counter printed above is
> there so you can check: it should always be two per block.

## 1.5 Training

The loop is the standard one: get a batch, compute the loss, backpropagate, step the
optimizer. Two details worth noting.

**`torch.autocast(bfloat16)`** runs the matrix multiplies in 16-bit while keeping the master
weights in 32-bit. This is already low precision -- BF16 is the baseline everything in this
workshop is measured against, not FP32.

**Batch 24 x context 1024 is about 25,000 tokens per step.** That is not an arbitrary
choice: below roughly this size the GPU is not busy enough for Transformer Engine's fusion
to pay for itself, and notebook 03's results invert. Notebook 00 section 0.5 measured the
same effect on the toy model.

**Throughput measurement starts after step 10.** The first steps pay for CUDA setup, kernel
selection and the allocator warming up; including them would measure startup, not speed.
This is the same discipline as notebook 00 section 0.3.

In [6]:
def train(model, cfg, steps=600, batch_size=24, lr=6e-4, log_every=100):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95),
                            weight_decay=0.1, fused=True)
    model.train()
    history, t0, tokens = [], None, 0

    for step in range(steps):
        x, y = get_batch("train", cfg, batch_size)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if step == 9:                                  # begin timing after warmup
            torch.cuda.synchronize()
            t0, tokens = time.perf_counter(), 0
        if step >= 10:
            tokens += x.numel()
        if step % log_every == 0 or step == steps - 1:
            torch.cuda.synchronize()
            tps = tokens / (time.perf_counter() - t0) if t0 else float("nan")
            history.append({"step": step, "loss": loss.item(), "tok_per_s": tps})
            print(f"  step {step:4d}   loss {loss.item():.3f}   {tps:9,.0f} tok/s")
    return history


torch.manual_seed(1337)
model = MiniGPT(cfg).cuda()
print("training:")
history = train(model, cfg)
baseline_tps = history[-1]["tok_per_s"]
print(f"\nBASELINE: {baseline_tps:,.0f} tokens/sec")

training:
  step    0   loss 11.028         nan tok/s
  step  100   loss 4.275     280,576 tok/s
  step  200   loss 3.864     274,768 tok/s
  step  300   loss 3.796     258,545 tok/s
  step  400   loss 3.485     241,128 tok/s
  step  500   loss 3.233     227,974 tok/s
  step  599   loss 3.002     218,355 tok/s

BASELINE: 218,355 tokens/sec


## 1.6 Talk to it

Loss numbers tell you whether training worked. Samples tell you whether it learned anything
you recognize.

Generation is the training objective run backwards: predict the next token, append it to the
input, repeat. `temperature` below 1 makes the model play it safe; `top_k` restricts each
choice to the k most likely tokens, cutting off the long tail of nonsense.

In [7]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=60, temperature=0.8, top_k=200):
    model.eval()
    ids = torch.tensor([enc.encode(prompt)], device="cuda")
    with torch.autocast("cuda", dtype=torch.bfloat16):
        for _ in range(max_new_tokens):
            logits, _ = model(ids[:, -model.cfg.seq_len:])
            logits = logits[:, -1, :] / temperature
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")
            ids = torch.cat([ids, torch.multinomial(F.softmax(logits, dim=-1), 1)], dim=1)
    model.train()
    return enc.decode(ids[0].tolist())


for prompt in ["Once upon a time, Mia built a small robot",
               "Tom and Lily found a key in the garden"]:
    print(generate(model, prompt), "\n")

Once upon a time, Mia built a small robot. The room, but Tim wanted to help the store.
One day, Bob saw Bob was scared and asked Tim, "I want to help you, let's test, you can help others." Tom. They played and had to go home. They did not see playing games.
Tim 

Tom and Lily found a key in the garden. She said, "You could print the cake."
Amy went to the store and wanted to try some more. They played the big party. They saw the sun. It was cold. Many toys. They started to fight and ran to the window and tug under the fence.
"Hello 



A few hundred steps is enough for grammar, names and the shape of a story, and not
enough for the plot to hold together. That is expected: this is roughly 2.5M tokens of
training where GPT-2 saw billions. What matters is that it is recognizably *stories* -- so
when a later change breaks something, you will see it.

## 1.7 What to measure, and what comes next

You now have two numbers that every later notebook reports:

- **loss** -- has quality changed?
- **tokens/sec** -- has speed changed?

Keep them separate. A change that makes training 20% faster and the loss 0.05 worse is not
obviously good or bad; it depends on your situation. Collapsing both into one "better"
number hides the trade.

Notebook 00 predicted what to expect here. The four matrix multiplies per block are
`768 x 2304`, `768 x 768`, `768 x 3072` and `3072 x 768`. Look back at your sweep in section
0.4: at those widths, FP8 was around or just past break-even -- so expect a real but modest
speedup, not the 1.5x you measured at `H = 8192`.

**Notebook 02** replaces the four `nn.Linear` layers with Transformer Engine equivalents,
which unlocks FP8. **Notebook 03** lets TE fuse the LayerNorms into the matmuls that follow
them. Both change only `TorchBlock`; everything else in this notebook stays exactly as it is.

## Exercises

1. **Check the prediction.** Compute the four GEMM shapes per block for `n_embd=768`. Using
   notebook 00's sweep, what speedup would you predict from FP8 alone? Write it down now and
   compare after notebook 02.
2. **Break the alignment.** Set `vocab_size = 50257` and re-run the training cell. How much
   throughput does the unaligned output layer cost?
3. **Remove the residual scaling.** Set `RESIDUAL = ()` so every weight gets std 0.02, and
   train again from the same seed. How much does the final loss change?
4. **Tokens per step matter.** Halve `batch_size` to 12 and re-run. Throughput barely moves
   -- but notebooks 02 and 03 measure *much* smaller gains at that size. Why would giving
   the GPU less work per step shrink the benefit of a faster matmul? (Notebook 00 section
   0.5 has the answer.)
5. **Watch it learn.** Generate from the model after 50, 200 and 400 steps. What does it get
   right first -- spelling, grammar, or plot?